# 01_eda_datasets.ipynb

Primer notebook del proyecto **Sistema híbrido para predecir churn de clientes en retail y generar acciones personalizadas de retención mediante IA generativa**.

Objetivos de este notebook:
- Cargar los 3 archivos `.xlsx` de la carpeta `data/raw/`
- Inspeccionar estructura, hojas, columnas y tipos de datos
- Evaluar calidad inicial de datos
- Identificar la mejor base para definir churn y construir el dataset analítico

## 0. Librerías

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import openpyxl

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 120)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Configuración de rutas

Asegúrate de guardar los tres archivos Excel dentro de `data/raw/`.

In [3]:
RAW_PATH = Path('/content/drive/MyDrive/PI_LV/data/raw')
excel_files = sorted(list(RAW_PATH.glob('*.xlsx')))

print('Archivos encontrados:')
for f in excel_files:
    print('-', f.name)

Archivos encontrados:
- 0_PI_Dims-Deuda - ENVIAR.xlsx
- 0_PI_Ventas-2022-2023 - ENVIAR.xlsx
- 0_PI_Ventas-2024-2025 - ENVIAR.xlsx


## 2. Revisar hojas disponibles en cada archivo

In [4]:
excel_sheets = {}

for file in excel_files:
    xls = pd.ExcelFile(file)
    excel_sheets[file.name] = xls.sheet_names
    print(f'\n{file.name}')
    print(xls.sheet_names)


0_PI_Dims-Deuda - ENVIAR.xlsx
['Deudas', 'Dim_Gerencia', 'Dim_UnidadNegocio', 'Dim_TipoGerencia', 'Dim_Procedencia', 'Dim_Producto', 'Dim_Cliente']

0_PI_Ventas-2022-2023 - ENVIAR.xlsx
['Ventas_2023', 'Ventas_2022']

0_PI_Ventas-2024-2025 - ENVIAR.xlsx
['Ventas_2025-2', 'Ventas_2025-1', 'Ventas_2024']


## 3. Cargar las hojas de cada archivo

In [5]:
datasets = {}

for file in excel_files:
    xls = pd.ExcelFile(file)
    datasets[file.stem] = {}

    for sheet in xls.sheet_names:
        df = pd.read_excel(file, sheet_name=sheet)
        datasets[file.stem][sheet] = df
        print(f"{file.name} | {sheet}: {df.shape}")

0_PI_Dims-Deuda - ENVIAR.xlsx | Deudas: (454679, 28)
0_PI_Dims-Deuda - ENVIAR.xlsx | Dim_Gerencia: (8, 2)
0_PI_Dims-Deuda - ENVIAR.xlsx | Dim_UnidadNegocio: (119, 7)
0_PI_Dims-Deuda - ENVIAR.xlsx | Dim_TipoGerencia: (6, 2)
0_PI_Dims-Deuda - ENVIAR.xlsx | Dim_Procedencia: (4, 2)
0_PI_Dims-Deuda - ENVIAR.xlsx | Dim_Producto: (6073, 7)
0_PI_Dims-Deuda - ENVIAR.xlsx | Dim_Cliente: (167550, 25)
0_PI_Ventas-2022-2023 - ENVIAR.xlsx | Ventas_2023: (873613, 31)
0_PI_Ventas-2022-2023 - ENVIAR.xlsx | Ventas_2022: (843446, 31)
0_PI_Ventas-2024-2025 - ENVIAR.xlsx | Ventas_2025-2: (671019, 31)
0_PI_Ventas-2024-2025 - ENVIAR.xlsx | Ventas_2025-1: (583137, 31)
0_PI_Ventas-2024-2025 - ENVIAR.xlsx | Ventas_2024: (1002662, 31)


## 4. Vista rápida de cada dataset

In [6]:
for name, sheet_dict in datasets.items():
    print('='*90)
    print(f'DATASET: {name}')
    print('='*90)

    for sheet_name, df in sheet_dict.items():
        print(f'\nSHEET: {sheet_name}')
        display(df.head())
        print('Columnas:')
        print(df.columns.tolist())
        print('\nTipos de datos:')
        print(df.dtypes)
        print('\n')

DATASET: 0_PI_Dims-Deuda - ENVIAR

SHEET: Deudas


,Cliente_ID-NEW,Domicilio_ID,Cliente_Dom_ID-NEW,Vendedor_ID,Gerencia_ID,UnidadNegocio_ID,TipoGerencia_ID,Procedencia_ID,Periodo_ID,Documento_ID-NEW,TipoDocumento,FechaEmision,FechaVencimiento,DiasVencimiento,TerminoPago_Resumen,Dias,TerminoPago,Fecha_UltPago,Actualizacion,Moneda,Calificacion,DiasVencimientoMayor,OrdenCalificacion,Saldo,MontoDocumento,Monto_UltPago,Importe_FC,Saldo_FC
0,C138097,01,C138097-01,125,265.0,U111,579,585,2024-02-29,2024351407,FACTURA,2024-02-29,2024-04-29,000-Corriente,Crédito,-60,CREDITO 60 DIAS,NaT,2024-03-05 18:58:46.991,S/,NaN,NaN,99,2497.41,2497.41,0.0,0.0,0.0
1,C138097,01,C138097-01,125,265.0,U111,579,585,2024-02-29,2024351406,FACTURA,2024-02-29,2024-04-29,000-Corriente,Crédito,-60,CREDITO 60 DIAS,NaT,2024-03-05 18:58:46.991,S/,NaN,NaN,99,2022.17,2022.17,0.0,0.0,0.0
2,C062797,01,C062797-01,125,265.0,U111,579,585,2024-02-29,2024351000,FACTURA,2024-02-29,2024-04-29,000-Corriente,Crédito,-60,CREDITO 60 DIAS,NaT,2024-03-05 18:58:46.991,S/,NaN,NaN,99,402.23,402.23,0.0,0.0,0.0
3,C085739,01,C085739-01,125,265.0,U111,579,585,2024-02-29,2024350998,FACTURA,2024-02-29,2024-04-29,000-Corriente,Crédito,-60,CREDITO 60 DIAS,NaT,2024-03-05 18:58:46.991,S/,NaN,NaN,99,6821.04,6821.04,0.0,0.0,0.0
4,C113475,01,C113475-01,125,265.0,U111,579,585,2024-02-29,2024350567,FACTURA,2024-02-29,2024-04-29,000-Corriente,Crédito,-60,CREDITO 60 DIAS,NaT,2024-03-05 18:58:46.991,S/,NaN,NaN,99,4940.45,4940.45,0.0,0.0,0.0


Columnas:
['Cliente_ID-NEW', 'Domicilio_ID', 'Cliente_Dom_ID-NEW', 'Vendedor_ID', 'Gerencia_ID', 'UnidadNegocio_ID', 'TipoGerencia_ID', 'Procedencia_ID', 'Periodo_ID', 'Documento_ID-NEW', 'TipoDocumento', 'FechaEmision', 'FechaVencimiento', 'DiasVencimiento', 'TerminoPago_Resumen', 'Dias', 'TerminoPago', 'Fecha_UltPago', 'Actualizacion', 'Moneda', 'Calificacion', 'DiasVencimientoMayor', 'OrdenCalificacion', 'Saldo', 'MontoDocumento', 'Monto_UltPago', 'Importe_FC', 'Saldo_FC']

Tipos de datos:
Cliente_ID-NEW                  object
Domicilio_ID                    object
Cliente_Dom_ID-NEW              object
Vendedor_ID                      int64
Gerencia_ID                    float64
UnidadNegocio_ID                object
TipoGerencia_ID                  int64
Procedencia_ID                   int64
Periodo_ID              datetime64[ns]
Documento_ID-NEW                 int64
TipoDocumento                   object
FechaEmision            datetime64[ns]
FechaVencimiento        datetime64

,Gerencia_ID,Gerencia
0,2144,ROMANO FIERRO MANUEL ALBERTO
1,4148,HERBOZO IBAÑEZ GUSTAVO ENRIQUE
2,777,INTERNACIONAL
3,266,NORTE GERENCIA
4,1990,CENTRO ORIENTE GERENCIA


Columnas:
['Gerencia_ID', 'Gerencia']

Tipos de datos:
Gerencia_ID     int64
Gerencia       object
dtype: object



SHEET: Dim_UnidadNegocio


...

Columnas:
['Documento', 'Documento_ID-NEW', 'FechaEmision', 'TipoVenta', 'TerminoPago', 'TerminoPago_Resumen', 'Vendedor_ID', 'Gerencia_ID', 'UnidadNegocio_ID', 'TipoGerencia_ID', 'Procedencia_ID', 'Producto_ID-NEW', 'Cliente_ID-NEW', 'Domicilio_ID', 'Cliente_Dom_ID-NEW', 'Indicador_Impuesto', 'Moneda', 'Tipo_Cambio', 'Descuento_Porc', 'Almacen', 'Motivo_Descuento', 'FechaVencimiento', 'Galones', 'Cantidad', 'Total_con_Impuesto', 'Total_sin_Impuesto', 'Total_Costo', 'Total_Promocion', 'Total_Promocion_PV', 'Total_Total_Costo', 'Descuento']

Tipos de datos:
Documento                      object
Documento_ID-NEW                int64
FechaEmision           datetime64[ns]
TipoVenta                      object
TerminoPago                    object
TerminoPago_Resumen            object
Vendedor_ID                     int64
Gerencia_ID                     int64
UnidadNegocio_ID               object
TipoGerencia_ID                 int64
Procedencia_ID                  int64
Producto_ID-NEW    

,Documento,Documento_ID-NEW,FechaEmision,TipoVenta,TerminoPago,TerminoPago_Resumen,Vendedor_ID,Gerencia_ID,UnidadNegocio_ID,TipoGerencia_ID,Procedencia_ID,Producto_ID-NEW,Cliente_ID-NEW,Domicilio_ID,Cliente_Dom_ID-NEW,Indicador_Impuesto,Moneda,Tipo_Cambio,Descuento_Porc,Almacen,Motivo_Descuento,FechaVencimiento,Galones,Cantidad,Total_con_Impuesto,Total_sin_Impuesto,Total_Costo,Total_Promocion,Total_Promocion_PV,Total_Total_Costo,Descuento
0,Factura,2025000002,2025-06-30,Venta Neta,CREDITO 60 DIAS,Crédito,226,267,U301,579,585,P004204,C084376,1,C084376-01,IGV,S/,3.552,0.0,AN001,NINGUNO,2025-08-29,4.7520,3,452.62,383.58,173.090001,0.0,0.0,173.090001,0.0000
1,Factura,2025000003,2025-06-30,Venta Neta,CREDITO 60 DIAS,Crédito,226,267,U301,579,585,P002358,C084376,1,C084376-01,IGV,S/,3.552,8.0,AN001,NINGUNO,2025-08-29,2.9568,2,273.14,231.47,119.370000,0.0,0.0,119.370000,20.1280
2,Factura,2025000003,2025-06-30,Venta Neta,CREDITO 60 DIAS,Crédito,226,267,U301,579,585,P004282,C084376,1,C084376-01,IGV,S/,3.552,4.0,AN001,NINGUNO,2025-08-29,2.8152,3,285.47,241.92,84.770001,0.0,0.0,84.770001,10.0800
3,Factura,2025000003,2025-06-30,Venta Neta,CREDITO 60 DIAS,Crédito,226,267,U301,579,585,P004349,C084376,1,C084376-01,IGV,S/,3.552,7.0,AN001,NINGUNO,2025-08-29,0.8280,3,270.39,229.14,79.809999,0.0,0.0,79.809999,17.2473
4,Factura,2025000004,2025-06-30,Venta Neta,PAGO ADELANTADO,Contado,226,267,U301,579,585,P004204,C139073,1,C139073-01,IGV,S/,3.552,0.0,AN001,NINGUNO,2025-06-30,1.5840,1,143.70,121.78,57.690000,0.0,0.0,57.690000,0.0000


Columnas:
['Documento', 'Documento_ID-NEW', 'FechaEmision', 'TipoVenta', 'TerminoPago', 'TerminoPago_Resumen', 'Vendedor_ID', 'Gerencia_ID', 'UnidadNegocio_ID', 'TipoGerencia_ID', 'Procedencia_ID', 'Producto_ID-NEW', 'Cliente_ID-NEW', 'Domicilio_ID', 'Cliente_Dom_ID-NEW', 'Indicador_Impuesto', 'Moneda', 'Tipo_Cambio', 'Descuento_Porc', 'Almacen', 'Motivo_Descuento', 'FechaVencimiento', 'Galones', 'Cantidad', 'Total_con_Impuesto', 'Total_sin_Impuesto', 'Total_Costo', 'Total_Promocion', 'Total_Promocion_PV', 'Total_Total_Costo', 'Descuento']

Tipos de datos:
Documento                      object
Documento_ID-NEW                int64
FechaEmision           datetime64[ns]
TipoVenta                      object
TerminoPago                    object
TerminoPago_Resumen            object
Vendedor_ID                     int64
Gerencia_ID                     int64
UnidadNegocio_ID               object
TipoGerencia_ID                 int64
Procedencia_ID                  int64
Producto_ID-NEW    

,Documento,Documento_ID-NEW,FechaEmision,TipoVenta,TerminoPago,TerminoPago_Resumen,Vendedor_ID,Gerencia_ID,UnidadNegocio_ID,TipoGerencia_ID,Procedencia_ID,Producto_ID-NEW,Cliente_ID-NEW,Domicilio_ID,Cliente_Dom_ID-NEW,Indicador_Impuesto,Moneda,Tipo_Cambio,Descuento_Porc,Almacen,Motivo_Descuento,FechaVencimiento,Galones,Cantidad,Total_con_Impuesto,Total_sin_Impuesto,Total_Costo,Total_Promocion,Total_Promocion_PV,Total_Total_Costo,Descuento
0,Factura,2024000002,2024-12-31,Promoción,Contado,Contado,1956,265,U111,579,585,P004356,C144814,1,C144814-01,EXE_IGV,S/,3.77,0.0,AN001,NINGUNO,2024-12-31,30.0000,5,164.85,0.00,0.000000,103.59,164.85,103.590000,0.00000
1,Factura,2024000002,2024-12-31,Venta Neta,Contado,Contado,1956,265,U111,579,585,P004356,C144814,1,C144814-01,IGV,S/,3.77,0.0,AN001,NINGUNO,2024-12-31,90.0000,15,583.57,494.55,310.780005,0.00,0.00,310.780005,0.00000
2,Factura,2024000003,2024-12-31,Venta Neta,Contado,Contado,1143,265,U111,579,585,P002569,C077502,1,C077502-01,IGV,S/,3.77,11.8,AN001,NINGUNO,2024-12-31,1.7088,1,111.91,94.84,48.480000,0.00,0.00,48.480000,12.68854
3,Factura,2024000004,2024-12-31,Venta Neta,Contado,Contado,35,265,U111,579,585,P004299,C123767,1,C123767-01,IGV,S/,3.77,2.0,AN001,NINGUNO,2024-12-31,19.9384,4,840.15,711.99,409.180000,0.00,0.00,409.180000,14.53040
4,Factura,2024000004,2024-12-31,Promoción,Contado,Contado,35,265,U111,579,585,P004299,C123767,1,C123767-01,EXE_IGV,S/,3.77,0.0,AN001,NINGUNO,2024-12-31,4.9846,1,181.63,0.00,0.000000,102.29,181.63,102.290000,0.00000


Columnas:
['Documento', 'Documento_ID-NEW', 'FechaEmision', 'TipoVenta', 'TerminoPago', 'TerminoPago_Resumen', 'Vendedor_ID', 'Gerencia_ID', 'UnidadNegocio_ID', 'TipoGerencia_ID', 'Procedencia_ID', 'Producto_ID-NEW', 'Cliente_ID-NEW', 'Domicilio_ID', 'Cliente_Dom_ID-NEW', 'Indicador_Impuesto', 'Moneda', 'Tipo_Cambio', 'Descuento_Porc', 'Almacen', 'Motivo_Descuento', 'FechaVencimiento', 'Galones', 'Cantidad', 'Total_con_Impuesto', 'Total_sin_Impuesto', 'Total_Costo', 'Total_Promocion', 'Total_Promocion_PV', 'Total_Total_Costo', 'Descuento']

Tipos de datos:
Documento                      object
Documento_ID-NEW                int64
FechaEmision           datetime64[ns]
TipoVenta                      object
TerminoPago                    object
TerminoPago_Resumen            object
Vendedor_ID                     int64
Gerencia_ID                     int64
UnidadNegocio_ID               object
TipoGerencia_ID                 int64
Procedencia_ID                  int64
Producto_ID-NEW    

## 5. Resumen estructural

Este bloque ayuda a comparar rápidamente los tres archivos.

In [7]:
summary = []

for dataset_name, sheet_dict in datasets.items():
    for sheet_name, df in sheet_dict.items():
        summary.append({
            'dataset': dataset_name,
            'sheet': sheet_name,
            'rows': df.shape[0],
            'cols': df.shape[1],
            'missing_cells': int(df.isna().sum().sum()),
            'duplicate_rows': int(df.duplicated().sum())
        })

summary_df = pd.DataFrame(summary)
summary_df

,dataset,sheet,rows,cols,missing_cells,duplicate_rows
0,0_PI_Dims-Deuda - ENVIAR,Deudas,454679,28,973569,69
1,0_PI_Dims-Deuda - ENVIAR,Dim_Gerencia,8,2,0,0
2,0_PI_Dims-Deuda - ENVIAR,Dim_UnidadNegocio,119,7,124,0
3,0_PI_Dims-Deuda - ENVIAR,Dim_TipoGerencia,6,2,0,0
4,0_PI_Dims-Deuda - ENVIAR,Dim_Procedencia,4,2,0,0
5,0_PI_Dims-Deuda - ENVIAR,Dim_Producto,6073,7,3706,0
6,0_PI_Dims-Deuda - ENVIAR,Dim_Cliente,167550,25,161820,0
7,0_PI_Ventas-2022-2023 - ENVIAR,Ventas_2023,873613,31,364576,11348
8,0_PI_Ventas-2022-2023 - ENVIAR,Ventas_2022,843446,31,908723,8135
9,0_PI_Ventas-2024-2025 - ENVIAR,Ventas_2025-2,671019,31,93990,2078


## 6. Calidad de datos por dataset

In [8]:
for dataset_name, sheet_dict in datasets.items():
    print('='*90)
    print(f'CALIDAD DE DATOS: {dataset_name}')
    print('='*90)

    for sheet_name, df in sheet_dict.items():
        print(f'\nSHEET: {sheet_name}')
        quality = pd.DataFrame({
            'column': df.columns,
            'dtype': df.dtypes.astype(str).values,
            'nulls': df.isna().sum().values,
            'null_pct': (df.isna().mean().values * 100).round(2),
            'n_unique': df.nunique(dropna=True).values
        }).sort_values(by='null_pct', ascending=False)

        display(quality)


CALIDAD DE DATOS: 0_PI_Dims-Deuda - ENVIAR

SHEET: Deudas


,column,dtype,nulls,null_pct,n_unique
21,DiasVencimientoMayor,object,371265,81.65,9
17,Fecha_UltPago,datetime64[ns],322641,70.96,905
20,Calificacion,object,265425,58.38,7
16,TerminoPago,object,9961,2.19,12
4,Gerencia_ID,float64,3795,0.83,7
2,Cliente_Dom_ID-NEW,object,481,0.11,40331
6,TipoGerencia_ID,int64,0,0.00,1
7,Procedencia_ID,int64,0,0.00,1
3,Vendedor_ID,int64,0,0.00,417
5,UnidadNegocio_ID,object,0,0.00,74



SHEET: Dim_Gerencia


,column,dtype,nulls,null_pct,n_unique
0,Gerencia_ID,int64,0,0.0,8
1,Gerencia,object,0,0.0,8



SHEET: Dim_UnidadNegocio


...

,column,dtype,nulls,null_pct,n_unique
11,Producto_ID-NEW,object,48354,7.21,985
21,FechaVencimiento,datetime64[ns],44866,6.69,260
19,Almacen,object,770,0.11,33
3,TipoVenta,object,0,0.00,3
4,TerminoPago,object,0,0.00,12
1,Documento_ID-NEW,int64,0,0.00,285799
0,Documento,object,0,0.00,2
6,Vendedor_ID,int64,0,0.00,243
7,Gerencia_ID,int64,0,0.00,6
9,TipoGerencia_ID,int64,0,0.00,1



SHEET: Ventas_2025-1


,column,dtype,nulls,null_pct,n_unique
11,Producto_ID-NEW,object,72858,12.49,908
21,FechaVencimiento,datetime64[ns],35302,6.05,254
19,Almacen,object,210,0.04,39
3,TipoVenta,object,0,0.00,3
4,TerminoPago,object,0,0.00,11
1,Documento_ID-NEW,int64,0,0.00,230051
0,Documento,object,0,0.00,2
6,Vendedor_ID,int64,0,0.00,235
7,Gerencia_ID,int64,0,0.00,7
9,TipoGerencia_ID,int64,0,0.00,1



SHEET: Ventas_2024


,column,dtype,nulls,null_pct,n_unique
11,Producto_ID-NEW,object,83849,8.36,856
21,FechaVencimiento,datetime64[ns],56431,5.63,443
19,Almacen,object,606,0.06,35
3,TipoVenta,object,0,0.00,3
4,TerminoPago,object,0,0.00,15
1,Documento_ID-NEW,int64,0,0.00,405389
0,Documento,object,0,0.00,2
6,Vendedor_ID,int64,0,0.00,203
7,Gerencia_ID,int64,0,0.00,8
9,TipoGerencia_ID,int64,0,0.00,1


## 7. Identificación preliminar del rol de cada archivo

- **0_PI_Ventas-2022-2023 - ENVIAR.xlsx**: contiene información transaccional de ventas por año (2022 y 2023). Su rol principal es servir como fuente histórica de comportamiento de compra.
- **0_PI_Ventas-2024-2025 - ENVIAR.xlsx**: contiene información transaccional de ventas por año y corte (2024, 2025-1 y 2025-2). Complementa la serie temporal de ventas y será parte de la base principal para construir churn.
- **0_PI_Dims-Deuda - ENVIAR.xlsx**: contiene una tabla de hechos (`Deudas`) y varias dimensiones (`Dim_Cliente`, `Dim_Producto`, `Dim_Gerencia`, etc.). Su rol principal es enriquecer el análisis con atributos del cliente, producto y situación de deuda.

### Clasificación preliminar
- Archivos de ventas: **transacciones**
- Archivo de deuda: **hechos + dimensiones**
- Unidad objetivo del proyecto: **cliente**

## 8. Criterios para definir churn

1. **¿Existe una fecha de transacción o última compra?**  
   Preliminarmente sí, ya que las ventas están organizadas por periodos anuales y semestrales, lo que sugiere la existencia de variables temporales de operación o compra. Esto debe confirmarse con la revisión de columnas.

2. **¿La unidad es cliente o transacción?**  
   La fuente principal de ventas parece estar a nivel transaccional, pero el dataset analítico final se construirá a nivel cliente.

3. **¿Se puede calcular recencia, frecuencia y monto?**  
   Sí, si las tablas de ventas contienen identificador de cliente, fecha de compra y monto vendido. Estas tres variables permitirán construir métricas RFM.

4. **¿Qué ventana de inactividad sería razonable?**  
   Como hipótesis inicial, se propone usar una ventana de **90 días sin compra** para definir churn. Esta regla podrá ajustarse según la frecuencia real de compra observada en el EDA.

### Criterio preliminar de churn
Se definirá churn como la ausencia de compras del cliente durante una ventana determinada de tiempo, usando la consolidación histórica de ventas como fuente principal.

## 9. Próximo paso

Después de este EDA inicial, el siguiente paso será construir el dataset analítico para modelado.

### Decisiones preliminares
- La base principal del proyecto será la consolidación de las hojas de ventas 2022, 2023, 2024, 2025-1 y 2025-2.
- Las columnas clave para definir churn serán las relacionadas con:
  - identificador de cliente
  - fecha de compra o fecha de operación
  - monto de venta
- El archivo de deuda se utilizará como fuente complementaria para enriquecer atributos del cliente.
- Será necesario identificar la llave común entre ventas y las dimensiones de deuda, especialmente `Dim_Cliente`.

### Siguiente notebook
El siguiente notebook será `02_data_preparation_churn.ipynb`, donde se realizará:
- la unificación de hojas de ventas
- la identificación de la llave de cliente
- la construcción de variables RFM
- la definición operativa de churn

## 10. Conclusiones del EDA

- Los archivos de ventas constituyen la fuente principal para construir la variable churn.
- La unidad de análisis objetivo será el cliente.
- Es necesario consolidar las hojas de ventas 2022–2025 en una sola tabla transaccional.
- El archivo de deuda se utilizará como fuente complementaria para enriquecer variables del cliente.
- Se deberá confirmar la llave común entre ventas y las dimensiones del archivo de deuda, especialmente `Dim_Cliente`.
- El siguiente paso será construir un dataset analítico a nivel cliente y definir churn según inactividad de compra.